# Time Series Plot

The time series plot is the foundational visualisation in any time series workflow: a measured quantity drawn against an ordered time axis. This tutorial covers single-series line plots, multi-series overlays, and styling with [Plots.jl](https://github.com/JuliaPlots/Plots.jl).

Before fitting any model, you plot the data — trend, seasonality, level shifts, and outliers are almost always visible to the eye.

In [ ]:
Pollis.packages("Plots")

In [ ]:
using Plots
using Dates
using Random

Random.seed!(42);

## A Single Series

We build a synthetic monthly series with a trend, a seasonal component, and noise, indexed by real dates so the time axis is meaningful.

In [ ]:
n = 72
t = Date(2018, 1, 1) .+ Month.(0:n-1)

trend    = 0.15 .* (0:n-1)
seasonal = 3.0 .* sin.(2 * pi .* (0:n-1) ./ 12)
noise    = randn(n)
y = 10 .+ trend .+ seasonal .+ noise;

In [ ]:
plot(t, y;
    label  = "Observed",
    xlabel = "Date",
    ylabel = "Value",
    title  = "Monthly Series",
    lw     = 2,
    legend = :topleft)

Connecting consecutive observations with line segments is what encodes the *ordering* of a time series, distinguishing it from an unordered sample. The upward drift (trend) and the regular twelve-month wave (seasonality) are both immediately visible.

## Multi-Series Overlay

Several series can be drawn on shared axes to compare scale, co-movement, and timing. Use `plot!` to add to an existing plot.

In [ ]:
y2 = 10 .+ 0.15 .* (0:n-1) .+ 3.0 .* sin.(2 * pi .* ((0:n-1) .- 3) ./ 12) .+ randn(n)

plot(t, y;  label = "Series A", lw = 2, xlabel = "Date", ylabel = "Value")
plot!(t, y2; label = "Series B", lw = 2, linestyle = :dash)

Series B is constructed with a three-month phase shift. Overlaying the two makes the lead–lag relationship visible: B's peaks consistently arrive after A's.

### Differing scales

When series live on very different scales, plotting them on one axis collapses the smaller one to a flat line. Indexing each series to a common base (first value = 100) restores a fair comparison of *shape* rather than *magnitude*.

In [ ]:
index(x) = 100 .* x ./ x[1]

plot(t, index(y);  label = "Series A (indexed)", lw = 2, xlabel = "Date", ylabel = "Index (start = 100)")
plot!(t, index(y2); label = "Series B (indexed)", lw = 2, linestyle = :dash)

### Two native scales (dual axis)

When you need the *actual magnitudes* of both series rather than just their shape, give each its own y-axis: `twinx()` adds a second axis on the right. Read it with care — where the two lines cross depends on how each axis is scaled, so a dual axis can suggest a relationship that is not really there.

In [ ]:
plot(t, y;  label = "Series A (left)",  ylabel = "Series A", legend = :topleft, lw = 2)
plot!(twinx(), t, y2; label = "Series B (right)", ylabel = "Series B", legend = :topright,
      color = :red, lw = 2, linestyle = :dash)

## Styling

A few adjustments make a plot publication-ready: line width to emphasise the primary series, marker overlays to highlight individual observations, and grid and legend placement clear of the data.

In [ ]:
plot(t, y;
    label      = "Observed",
    lw         = 2.5,
    marker     = (:circle, 3),
    xlabel     = "Date",
    ylabel     = "Value",
    title      = "Styled Time Series",
    legend     = :topleft,
    grid       = true,
    framestyle = :box)

## Handling Missing Data

Missing observations break the continuity of a series. Representing them as `missing` lets Plots.jl render an honest gap rather than connecting distant points with a misleading straight segment.

In [ ]:
yg = Vector{Union{Missing, Float64}}(y)
yg[30:35] .= missing

plot(t, yg;
    label  = "With gap",
    lw     = 2,
    xlabel = "Date",
    ylabel = "Value",
    title  = "Missing Stretch Rendered as a Gap")

## Summary

- A line plot encodes the ordering of a time series and reveals trend, seasonality, breaks, and outliers at a glance.
- Overlays expose co-movement and lead–lag; normalise when scales differ.
- Keep `missing` values as `missing` so gaps render honestly.

Next, see the **Ribbon Plot** tutorial to add an uncertainty band around a forecast.